In [2]:
# Cell 1: Install dependencies (only need panns-inference, others are pre-installed on Colab)
%pip install -q panns-inference soundfile

In [3]:
# Cell 2: Verify GPU is available
import torch

print("=" * 50)
print("GPU Configuration")
print("=" * 50)

if torch.cuda.is_available():
    print(f"✅ CUDA available: True")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("Go to Runtime → Change runtime type → A100 GPU")


GPU Configuration
✅ CUDA available: True
✅ GPU: NVIDIA A100-SXM4-40GB
✅ VRAM: 42.5 GB


In [ ]:
# Cell 3: Clone repo + upload audio
import os
from google.colab import files

os.chdir("/content")

# Step 1: Clone from GitHub
if not os.path.exists("/content/TrumpetJudge"):
    print("📦 Cloning from GitHub...")
    !git clone https://github.com/AdnanKapadia/TrumpetJudge.git
    print("✅ Cloned!")
else:
    print("✅ Repo already cloned, pulling latest...")
    os.chdir("/content/TrumpetJudge")
    !git pull

os.chdir("/content/TrumpetJudge")

# Step 2: Upload audio.zip
if os.path.exists("data/audio") and len(os.listdir("data/audio")) > 10:
    print("✅ Audio files already present!")
else:
    print("\n" + "=" * 50)
    print("📁 Upload audio.zip")
    print("=" * 50)
    print("\nCreate it locally:")
    print("  cd /home/adnan/TrumpetJudge/data")
    print("  zip -r audio.zip audio/")
    print("\nThen click 'Choose Files' below:\n")
    
    uploaded = files.upload()
    !unzip -q -o audio.zip -d data/
    print("\n✅ Audio extracted!")

# Verify
print(f"\n📂 Working directory: {os.getcwd()}")
audio_count = len(os.listdir('data/audio')) if os.path.exists('data/audio') else 0
print(f"📁 Audio files: {audio_count}")

import pandas as pd
train_df = pd.read_csv("data/prepared/train.csv")
val_df = pd.read_csv("data/prepared/val.csv")
print(f"✅ Training samples: {len(train_df)}")
print(f"✅ Validation samples: {len(val_df)}")


📦 Cloning from GitHub...
Cloning into 'TrumpetJudge'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 210 (delta 16), reused 135 (delta 16), pack-reused 69 (from 2)
Receiving objects: 100% (210/210), 81.44 MiB | 15.89 MiB/s, done.
Resolving deltas: 100% (20/20), done.
✅ Cloned!

📁 Mounting Google Drive...


KeyboardInterrupt: 

In [ ]:
# Cell 4: Run Training! 🎺
# Using A100 optimized settings

!python ml/train.py \
    --train_csv data/prepared/train.csv \
    --val_csv data/prepared/val.csv \
    --augment \
    --batch_size 32 \
    --num_workers 2 \
    --epochs 50 \
    --patience 10

KeyboardInterrupt: 

In [ ]:
# Cell 5: View training results
import glob
import json

# Find latest checkpoint
checkpoint_dirs = sorted(glob.glob("checkpoints/run_*"))
if checkpoint_dirs:
    latest = checkpoint_dirs[-1]
    print(f"📁 Latest checkpoint: {latest}")
    
    # Load and display config
    with open(f"{latest}/config.json") as f:
        config = json.load(f)
    
    print(f"\n✅ Training completed!")
    print(f"   Best epoch: {config['best_epoch']}")
    print(f"   Best MAE: {config['best_val_mae']:.3f}")
    print(f"\n📦 Model saved to: {latest}/best_model.pt")
else:
    print("No checkpoints found yet - run training first!")


In [ ]:
# Cell 6: Download trained model (optional - for web Colab)
# Skip this if using VS Code extension - files are already local

from google.colab import files
import glob

checkpoint_dirs = sorted(glob.glob("checkpoints/run_*"))
if checkpoint_dirs:
    latest = checkpoint_dirs[-1]
    # Zip the checkpoint folder
    !zip -r trained_model.zip {latest}
    files.download("trained_model.zip")
    print(f"📥 Downloaded: trained_model.zip")
else:
    print("No checkpoints to download!")
